# Modelo Preditivo de Falhas Viárias - São Luís / MA
Este notebook consolida todo o fluxo de trabalho de Data Science: desde a limpeza dos dados até o treinamento do modelo XGBoost.

## 1. Extração e Limpeza de Dados (Data Wrangling)

In [ ]:
import pandas as pd
import numpy as np
"""
Projeto: Preditor de Falhas em Vias - São Luís / MA
Etapa:   Coleta, Limpeza e Transformação de Dados
Autor:   Data Science - Prefeitura de São Luís
"""

from datetime import datetime
warnings.filterwarnings("ignore")

DATA_DIR = "/mnt/user-data/outputs/sao_luis_road_data/"
OUT_DIR  = "/mnt/user-data/outputs/sao_luis_road_data/"

# UTILIDADES

def section(title):

def info(msg):   print(f"  ✔  {msg}")
def warn(msg):   print(f"  ⚠  {msg}")
def issue(msg):  print(f"  ✘  {msg}")

def quality_report(df, name):
    """Gera relatório de qualidade de dados."""
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        info("Sem valores nulos")
    else:
        for col, n in missing.items():
            pct = n / len(df) * 100
            warn(f"Nulos em '{col}': {n} ({pct:.1f}%)")
    dups = df.duplicated().sum()
    if dups:
        issue(f"{dups} linhas duplicadas encontradas")
    else:
        info("Sem duplicatas")

# ═══════════════════════════════════════════════════════════
# PASSO 1 — CARREGAR DADOS BRUTOS
# ═══════════════════════════════════════════════════════════

segments   = pd.read_csv(DATA_DIR + "01_road_segments.csv")
repairs    = pd.read_csv(DATA_DIR + "02_repair_history.csv")
rainfall   = pd.read_csv(DATA_DIR + "03_rainfall_data.csv")
traffic    = pd.read_csv(DATA_DIR + "04_traffic_load.csv")
complaints = pd.read_csv(DATA_DIR + "05_citizen_complaints.csv")

# ═══════════════════════════════════════════════════════════
# PASSO 2 — DIAGNÓSTICO DE QUALIDADE
# ═══════════════════════════════════════════════════════════

quality_report(segments,   "01_road_segments")
quality_report(repairs,    "02_repair_history")
quality_report(rainfall,   "03_rainfall_data")
quality_report(traffic,    "04_traffic_load")
quality_report(complaints, "05_citizen_complaints")

# ═══════════════════════════════════════════════════════════
# PASSO 3 — LIMPEZA: SEGMENTOS
# ═══════════════════════════════════════════════════════════

seg = segments.copy()

# Padronizar strings
for col in ["street_name", "bairro", "zone", "pavement_type", "road_class"]:
    seg[col] = seg[col].str.strip().str.lower()

# Corrigir tipo de pavimento (normalizar variações de escrita)
pav_map = {
    "asfalto": "asfalto",
    "asphalt": "asfalto",
    "paralelepipedo": "paralelepipedo",
    "paralelepípedo": "paralelepipedo",
    "terra": "terra",
    "concreto": "concreto",
}
seg["pavement_type"] = seg["pavement_type"].map(pav_map).fillna("desconhecido")

# Converter booleanos (sim/nao → 1/0)
for col in ["truck_route", "bus_route", "flood_zone"]:
    seg[col] = seg[col].str.lower().map({"sim": 1, "nao": 0, "yes": 1, "no": 0})

# Validar coordenadas (São Luís: lat ~-2.4 a -2.6 / lon ~-44.1 a -44.4)
lat_ok = seg["lat_start"].between(-2.7, -2.3) & seg["lat_end"].between(-2.7, -2.3)
lon_ok = seg["lon_start"].between(-44.5, -44.0) & seg["lon_end"].between(-44.5, -44.0)
invalid_coords = seg[~(lat_ok & lon_ok)]
if len(invalid_coords):
    issue(f"{len(invalid_coords)} segmentos com coordenadas fora de São Luís — verificar!")
else:
    info("Todas as coordenadas dentro dos limites de São Luís")

# Calcular idade da pavimentação
current_year = datetime.now().year
seg["pavement_age_years"] = current_year - seg["pavement_year"]
seg["years_since_overlay"] = seg["last_overlay_year"].apply(
    lambda y: current_year - y if pd.notna(y) else np.nan
)

# Codificar tipo de pavimento (para o modelo)
pav_code = {"asfalto": 1, "paralelepipedo": 2, "terra": 3, "concreto": 4, "desconhecido": 0}
seg["pavement_type_code"] = seg["pavement_type"].map(pav_code)

# ═══════════════════════════════════════════════════════════
# PASSO 3B — LIMPEZA: HISTÓRICO DE REPAROS
# ═══════════════════════════════════════════════════════════

rep = repairs.copy()

# Converter datas
rep["order_date"]      = pd.to_datetime(rep["order_date"],      errors="coerce")
rep["completion_date"] = pd.to_datetime(rep["completion_date"], errors="coerce")

# Detectar datas inválidas
invalid_dates = rep["order_date"].isna().sum()
if invalid_dates:
    issue(f"{invalid_dates} ordens com data inválida — removendo")
    rep = rep.dropna(subset=["order_date"])

# Calcular duração real do reparo
rep["repair_duration_days"] = (rep["completion_date"] - rep["order_date"]).dt.days

# Detectar durações absurdas (negativas ou > 60 dias)
outliers = rep[(rep["repair_duration_days"] < 0) | (rep["repair_duration_days"] > 60)]
if len(outliers):
    warn(f"{len(outliers)} reparos com duração suspeita (negativa ou >60 dias)")
    rep.loc[rep["repair_duration_days"] < 0, "repair_duration_days"] = np.nan

# Remover duplicatas exatas
before = len(rep)
rep = rep.drop_duplicates(subset=["segment_id", "order_date", "repair_type"])
removed = before - len(rep)
if removed:
    warn(f"{removed} reparos duplicados removidos")
else:
    info("Sem duplicatas em reparos")

# Padronizar tipo de reparo
rep["repair_type"] = rep["repair_type"].str.strip().str.lower()
rep["failure_type"] = rep["failure_type"].str.strip().str.lower()
rep["priority"]    = rep["priority"].str.strip().str.lower()

# Extrair ano/mês para joins temporais
rep["repair_year"]  = rep["order_date"].dt.year
rep["repair_month"] = rep["order_date"].dt.month

# Custo nulo → imputar com mediana do tipo de reparo
median_cost = rep.groupby("repair_type")["repair_cost_brl"].median()
rep["repair_cost_brl"] = rep.apply(
    lambda row: median_cost.get(row["repair_type"], rep["repair_cost_brl"].median())
    if pd.isna(row["repair_cost_brl"]) else row["repair_cost_brl"],
    axis=1
)

# ═══════════════════════════════════════════════════════════
# PASSO 3C — LIMPEZA: DADOS DE CHUVA
# ═══════════════════════════════════════════════════════════

rain = rainfall.copy()

# Validar valores de chuva (São Luís: 0–600mm/mês é razoável)
rain_outliers = rain[rain["rainfall_mm"] > 600]
if len(rain_outliers):
    warn(f"{len(rain_outliers)} meses com chuva >600mm — verificar sensor")

rain_negative = rain[rain["rainfall_mm"] < 0]
if len(rain_negative):
    issue(f"{len(rain_negative)} registros com chuva negativa — corrigindo para 0")
    rain.loc[rain["rainfall_mm"] < 0, "rainfall_mm"] = 0

# Criar coluna de data para ordenação e cálculos
rain["date"] = pd.to_datetime(rain[["year", "month"]].assign(day=1))
rain = rain.sort_values("date").reset_index(drop=True)

# Calcular precipitações acumuladas (janelas temporais)
rain["rainfall_30d"]  = rain["rainfall_mm"]
rain["rainfall_90d"]  = rain["rainfall_mm"].rolling(3,  min_periods=1).sum()
rain["rainfall_180d"] = rain["rainfall_mm"].rolling(6,  min_periods=1).sum()
rain["rainfall_365d"] = rain["rainfall_mm"].rolling(12, min_periods=1).sum()

# Identificar meses críticos (top quartil de chuva)
q75 = rain["rainfall_mm"].quantile(0.75)
rain["is_critical_month"] = (rain["rainfall_mm"] >= q75).astype(int)

# Preencher meses faltantes com interpolação linear
full_dates = pd.date_range(rain["date"].min(), rain["date"].max(), freq="MS")
rain = rain.set_index("date").reindex(full_dates)
numeric_cols = ["rainfall_mm", "rainy_days", "max_daily_mm",
                "extreme_events_50mm", "rainfall_30d",
                "rainfall_90d", "rainfall_180d", "rainfall_365d"]
rain[numeric_cols] = rain[numeric_cols].interpolate(method="linear")
rain = rain.reset_index().rename(columns={"index": "date"})
rain["year"]  = rain["date"].dt.year
rain["month"] = rain["date"].dt.month

# ═══════════════════════════════════════════════════════════
# PASSO 3D — LIMPEZA: TRÁFEGO
# ═══════════════════════════════════════════════════════════

traf = traffic.copy()

# Validar percentual de veículos pesados (0–100%)
invalid_pct = traf[~traf["heavy_vehicles_pct"].between(0, 100)]
if len(invalid_pct):
    issue(f"{len(invalid_pct)} registros com % de pesados inválido")

# Preencher load_index nulo com estimativa baseada em volume + pesados
mask = traf["load_index_1_10"].isna()
if mask.sum():
    traf.loc[mask, "load_index_1_10"] = (
        (traf.loc[mask, "avg_daily_vehicles"] / 2000 +
         traf.loc[mask, "heavy_vehicles_pct"] / 5)
        .clip(1, 10)
        .round()
    )
    warn(f"{mask.sum()} load_index imputados por fórmula")

# Para segmentos sem dado de tráfego, usar mediana por road_class
traf_latest = traf.sort_values("measurement_year").groupby("segment_id").last().reset_index()

# Merge com road_class para imputação
traf_latest = traf_latest.merge(
    seg[["segment_id", "road_class"]], on="segment_id", how="left"
)
median_load_by_class = traf_latest.groupby("road_class")["load_index_1_10"].median()

# ═══════════════════════════════════════════════════════════
# PASSO 3E — LIMPEZA: RECLAMAÇÕES
# ═══════════════════════════════════════════════════════════

comp = complaints.copy()

comp["report_date"]      = pd.to_datetime(comp["report_date"],      errors="coerce")
comp["resolution_date"]  = pd.to_datetime(comp["resolution_date"],  errors="coerce")

# Remover duplicatas (mesmo segmento, mesma data, mesmo tipo)
before = len(comp)
comp = comp.drop_duplicates(subset=["segment_id", "report_date", "complaint_type"])
removed = before - len(comp)
if removed:
    warn(f"{removed} reclamações duplicadas removidas (mesmo segmento/dia/tipo)")

# Padronizar severidade
sev_map = {"leve": 1, "moderado": 2, "grave": 3, "muito_grave": 4}
comp["severity_code"] = comp["severity_reported"].str.lower().map(sev_map).fillna(2)

# Preencher days_to_resolve calculando quando possível
mask = comp["days_to_resolve"].isna() & comp["resolution_date"].notna()
comp.loc[mask, "days_to_resolve"] = (
    (comp.loc[mask, "resolution_date"] - comp.loc[mask, "report_date"]).dt.days
)

comp["report_year"]  = comp["report_date"].dt.year
comp["report_month"] = comp["report_date"].dt.month

# ═══════════════════════════════════════════════════════════
# PASSO 4 — ENGENHARIA DE FEATURES
# ═══════════════════════════════════════════════════════════

# ── 4A: Agregações de reparo por segmento ──────────────────
rep_agg = rep.groupby("segment_id").agg(
    total_repairs          = ("repair_id",         "count"),
    total_repair_cost      = ("repair_cost_brl",   "sum"),
    avg_repair_cost        = ("repair_cost_brl",   "mean"),
    last_repair_year       = ("repair_year",        "max"),
    last_repair_month      = ("repair_month",       lambda x: rep.loc[x.index, "repair_month"].iloc[-1]),
    emergency_repairs      = ("priority",           lambda x: (x == "urgente").sum()),
    buraco_count           = ("failure_type",       lambda x: (x == "buraco").sum()),
    afundamento_count      = ("failure_type",       lambda x: (x == "afundamento").sum()),
).reset_index()

rep_agg["years_since_last_repair"] = current_year - rep_agg["last_repair_year"]
rep_agg["recurrence_rate"]         = rep_agg["total_repairs"] / (
    (current_year - 2019) + 1
)  # reparos por ano

# ── 4B: Agregações de reclamações por segmento ─────────────
comp_agg = comp.groupby("segment_id").agg(
    total_complaints       = ("complaint_id",   "count"),
    avg_severity           = ("severity_code",  "mean"),
    muito_grave_count      = ("severity_code",  lambda x: (x == 4).sum()),
    unique_years           = ("report_year",    "nunique"),
).reset_index()

# ── 4C: Features de chuva sazonais ─────────────────────────
rain_season = rain.copy()
rain_season["is_rainy_season"] = rain_season["month"].between(1, 6).astype(int)

avg_rainy    = rain_season[rain_season["is_rainy_season"] == 1]["rainfall_mm"].mean()
avg_dry      = rain_season[rain_season["is_rainy_season"] == 0]["rainfall_mm"].mean()
rain_ratio   = avg_rainy / avg_dry if avg_dry > 0 else 0

# ═══════════════════════════════════════════════════════════
# PASSO 5 — MONTAR DATASET FINAL
# ═══════════════════════════════════════════════════════════

# Base: todos os segmentos
final = seg[[
    "segment_id", "street_name", "bairro", "zone",
    "pavement_type", "pavement_type_code", "pavement_age_years",
    "years_since_overlay", "road_class",
    "flood_zone", "truck_route", "bus_route"
]].copy()

# Merge: tráfego
final = final.merge(
    traf_latest[["segment_id", "load_index_1_10", "avg_daily_vehicles",
                 "heavy_vehicles_pct", "traffic_class"]],
    on="segment_id", how="left"
)

# Imputar load_index faltante pela mediana da road_class
final["load_index_1_10"] = final.apply(
    lambda row: median_load_by_class.get(row["road_class"], 5)
    if pd.isna(row["load_index_1_10"]) else row["load_index_1_10"],
    axis=1
)

# Merge: histórico de reparos
final = final.merge(rep_agg, on="segment_id", how="left")
final["total_repairs"]          = final["total_repairs"].fillna(0).astype(int)
final["total_repair_cost"]      = final["total_repair_cost"].fillna(0)
final["emergency_repairs"]      = final["emergency_repairs"].fillna(0).astype(int)
final["buraco_count"]           = final["buraco_count"].fillna(0).astype(int)
final["afundamento_count"]      = final["afundamento_count"].fillna(0).astype(int)
final["recurrence_rate"]        = final["recurrence_rate"].fillna(0)
final["years_since_last_repair"]= final["years_since_last_repair"].fillna(
    final["pavement_age_years"]  # se nunca reparado, usar idade da pavimentação
)

# Merge: reclamações
final = final.merge(comp_agg, on="segment_id", how="left")
final["total_complaints"]  = final["total_complaints"].fillna(0).astype(int)
final["avg_severity"]      = final["avg_severity"].fillna(0)
final["muito_grave_count"] = final["muito_grave_count"].fillna(0).astype(int)

# Adicionar chuva mais recente disponível
latest_rain = rain.sort_values("date").iloc[-1]
final["current_rainfall_30d"]  = latest_rain["rainfall_30d"]
final["current_rainfall_90d"]  = latest_rain["rainfall_90d"]
final["current_rainfall_180d"] = latest_rain["rainfall_180d"]
final["current_month"]         = latest_rain["month"]
final["is_rainy_season"]       = int(latest_rain["month"] in range(1, 7))

# ── Score de Risco Composto (heurístico, pré-modelo) ───────
# Pesos calibrados para o contexto de São Luís:
# Chuva recente 30% + Idade pavimento 20% + Tráfego 20%
# + Histórico 15% + Reclamações 10% + Baixada 5%
def compute_risk_score(row):
    rain_score    = min(row["current_rainfall_30d"] / 500, 1.0) * 30
    age_score     = min(row["pavement_age_years"] / 30, 1.0) * 20
    traffic_score = min(row["load_index_1_10"] / 10, 1.0) * 20
    history_score = min(row["total_repairs"] / 5, 1.0) * 15
    complaint_score = min(row["total_complaints"] / 5, 1.0) * 10
    flood_score   = row["flood_zone"] * 5
    return round(rain_score + age_score + traffic_score +
                 history_score + complaint_score + flood_score, 1)

final["risk_score"] = final.apply(compute_risk_score, axis=1)

# Categoria de risco
def risk_category(score):
    if score >= 80: return "critico"
    if score >= 60: return "alto"
    if score >= 40: return "medio"
    return "baixo"

final["risk_category"] = final["risk_score"].apply(risk_category)
final["priority_rank"] = final["risk_score"].rank(ascending=False, method="min").fillna(0).astype(int)
final = final.sort_values("risk_score", ascending=False).reset_index(drop=True)

# ═══════════════════════════════════════════════════════════
# PASSO 6 — SALVAR OUTPUTS
# ═══════════════════════════════════════════════════════════

# Dataset analítico completo
final.to_csv(OUT_DIR + "08_analytical_dataset.csv", index=False)

# Top 10 segmentos de risco (para o dashboard)
top10 = final.head(10)[[
    "priority_rank", "segment_id", "street_name", "bairro",
    "risk_score", "risk_category",
    "pavement_age_years", "load_index_1_10", "flood_zone",
    "total_repairs", "total_complaints",
    "current_rainfall_30d"
]]
top10.to_csv(OUT_DIR + "09_top_risk_segments.csv", index=False)

# Relatório de qualidade de dados
quality_log = []
for df, name in [(seg, "segments"), (rep, "repairs"), (rain, "rainfall"),
                 (traf, "traffic"), (comp, "complaints")]:
    for col in df.columns:
        n_null = df[col].isnull().sum()
        if n_null > 0:
            quality_log.append({
                "file": name,
                "column": col,
                "null_count": n_null,
                "null_pct": round(n_null / len(df) * 100, 1)
            })

pd.DataFrame(quality_log).to_csv(OUT_DIR + "10_data_quality_log.csv", index=False)

# Sumário de custos por segmento
cost_summary = rep.groupby("segment_id").agg(
    n_repairs          = ("repair_id",       "count"),
    total_cost_brl     = ("repair_cost_brl", "sum"),
    avg_cost_brl       = ("repair_cost_brl", "mean"),
    max_cost_brl       = ("repair_cost_brl", "max"),
    n_emergency        = ("priority",        lambda x: (x == "urgente").sum()),
).reset_index().merge(seg[["segment_id", "street_name", "bairro"]], on="segment_id")
cost_summary.to_csv(OUT_DIR + "11_cost_summary.csv", index=False)

# ═══════════════════════════════════════════════════════════
# RESUMO EXECUTIVO
# ═══════════════════════════════════════════════════════════

total_spent = rep["repair_cost_brl"].sum()
avg_reactive = rep[rep["priority"] == "urgente"]["repair_cost_brl"].mean()
avg_preventive = rep[rep["priority"].isin(["media", "baixa"])]["repair_cost_brl"].mean()
savings_potential = avg_reactive - avg_preventive if avg_preventive else 0

  Custo total histórico de reparos:   R$ {total_spent:,.2f}
  Custo médio reparo EMERGENCIAL:     R$ {avg_reactive:,.2f}
  Custo médio reparo PREVENTIVO:      R$ {avg_preventive:,.2f}
  Potencial de economia por reparo:   R$ {savings_potential:,.2f}

  Segmento mais problemático:
    {final.iloc[0]['street_name']} ({final.iloc[0]['bairro']})
    Score de risco: {final.iloc[0]['risk_score']} — {final.iloc[0]['risk_category'].upper()}
    Reparos históricos: {final.iloc[0]['total_repairs']}
    Reclamações registradas: {final.iloc[0]['total_complaints']}

  Próximos passos:
    → Carregar 08_analytical_dataset.csv no modelo XGBoost
    → Validar top-10 segmentos com equipes de campo
    → Integrar alerta de chuva do CEMADEN via API
""")

## 2. Análise Exploratória (EDA)

In [ ]:
import pandas as pd
import numpy as np
"""
Projeto: Preditor de Falhas em Vias - São Luís / MA
Etapa:   Análise Exploratória de Dados (EDA)
"""
matplotlib.use('Agg')
from scipy import stats
warnings.filterwarnings("ignore")

# Configuração visual
plt.rcParams.update({'figure.figsize': (12, 7), 'font.size': 11,
    'axes.titlesize': 14, 'axes.labelsize': 12, 'figure.dpi': 150})
sns.set_style("whitegrid")
COLORS = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']

BASE = os.path.dirname(os.path.abspath(__file__))
PARENT = os.path.dirname(BASE)
OUT = os.path.join(BASE, "eda_outputs")
os.makedirs(OUT, exist_ok=True)

# ── Carregar dados ──
segments   = pd.read_csv(os.path.join(PARENT, "01_road_segments.csv"))
repairs    = pd.read_csv(os.path.join(PARENT, "02_repair_history.csv"))
rainfall   = pd.read_csv(os.path.join(PARENT, "03_rainfall_data.csv"))
traffic    = pd.read_csv(os.path.join(PARENT, "04_traffic_load.csv"))
complaints = pd.read_csv(os.path.join(PARENT, "05_citizen_complaints.csv"))
ml_data    = pd.read_csv(os.path.join(PARENT, "06_ml_training_dataset.csv"))
risk       = pd.read_csv(os.path.join(PARENT, "07_risk_scores.csv"))
analytical = pd.read_csv(os.path.join(BASE, "08_analytical_dataset.csv"))

repairs["order_date"] = pd.to_datetime(repairs["order_date"])
repairs["completion_date"] = pd.to_datetime(repairs["completion_date"])
complaints["report_date"] = pd.to_datetime(complaints["report_date"])


# ═══════════════════════════════════════════════════════════
# 1. ESTATÍSTICAS DESCRITIVAS
# ═══════════════════════════════════════════════════════════

desc = analytical[['pavement_age_years','load_index_1_10','avg_daily_vehicles',
    'heavy_vehicles_pct','total_repairs','total_repair_cost','total_complaints',
    'risk_score']].describe().round(2)
desc.to_csv(os.path.join(OUT, "01_descritivas.csv"))

# ═══════════════════════════════════════════════════════════
# 2. DISTRIBUIÇÃO DA VARIÁVEL ALVO (risk_score / risk_category)
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma do risk_score
axes[0].hist(analytical['risk_score'].dropna(), bins=10, color='#3498db',
    edgecolor='white', alpha=0.85)
axes[0].axvline(analytical['risk_score'].mean(), color='#e74c3c', ls='--',
    lw=2, label=f"Média: {analytical['risk_score'].mean():.1f}")
axes[0].set_xlabel('Risk Score'); axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do Score de Risco'); axes[0].legend()

# Contagem por categoria
cat_order = ['baixo','medio','alto']
cat_counts = analytical['risk_category'].value_counts().reindex(cat_order).fillna(0)
bars = axes[1].bar(cat_counts.index, cat_counts.values,
    color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white')
for b in bars:
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
        int(b.get_height()), ha='center', fontweight='bold')
axes[1].set_title('Segmentos por Categoria de Risco')
axes[1].set_ylabel('Quantidade')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "02_distribuicao_risco.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 3. ANÁLISE POR ZONA E BAIRRO
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

zone_risk = analytical.groupby('zone')['risk_score'].mean().sort_values(ascending=False)
axes[0].barh(zone_risk.index, zone_risk.values, color=COLORS[:len(zone_risk)])
axes[0].set_xlabel('Score de Risco Médio'); axes[0].set_title('Risco Médio por Zona')
for i, v in enumerate(zone_risk.values):
    axes[0].text(v+0.5, i, f'{v:.1f}', va='center', fontweight='bold')

bairro_risk = analytical.groupby('bairro')['risk_score'].mean().sort_values(ascending=False).head(8)
axes[1].barh(bairro_risk.index, bairro_risk.values, color=sns.color_palette("YlOrRd_r", len(bairro_risk)))
axes[1].set_xlabel('Score de Risco Médio'); axes[1].set_title('Top 8 Bairros por Risco')
for i, v in enumerate(bairro_risk.values):
    axes[1].text(v+0.5, i, f'{v:.1f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "03_risco_zona_bairro.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 4. TIPO DE PAVIMENTO vs RISCO
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pav_counts = analytical['pavement_type'].value_counts()
axes[0].pie(pav_counts, labels=pav_counts.index, autopct='%1.0f%%',
    colors=COLORS, startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Distribuição por Tipo de Pavimento')

pav_data = []
pav_labels = []
for pt in analytical['pavement_type'].unique():
    subset = analytical[analytical['pavement_type']==pt]['risk_score'].dropna()
    if len(subset) > 0:
        pav_data.append(subset.values)
        pav_labels.append(pt)
bp = axes[1].boxplot(pav_data, labels=pav_labels,
    patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title('Score de Risco por Tipo de Pavimento')
axes[1].set_ylabel('Risk Score')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "04_pavimento_risco.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 5. CORRELAÇÃO ENTRE VARIÁVEIS NUMÉRICAS
# ═══════════════════════════════════════════════════════════
num_cols = ['pavement_age_years','load_index_1_10','avg_daily_vehicles',
    'heavy_vehicles_pct','total_repairs','total_repair_cost',
    'total_complaints','avg_severity','flood_zone','risk_score']
corr = analytical[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
    center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Matriz de Correlação — Variáveis Preditoras vs Risk Score')
plt.tight_layout()
plt.savefig(os.path.join(OUT, "05_correlacao.png"), bbox_inches='tight')
plt.close()

corr_target = corr['risk_score'].drop('risk_score').sort_values(ascending=False)
for col, val in corr_target.items():

# ═══════════════════════════════════════════════════════════
# 6. ANÁLISE TEMPORAL — CHUVAS E REPAROS
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

rainfall['date'] = pd.to_datetime(rainfall[['year','month']].assign(day=1))
axes[0].fill_between(rainfall['date'], rainfall['rainfall_mm'], alpha=0.3, color='#3498db')
axes[0].plot(rainfall['date'], rainfall['rainfall_mm'], color='#2980b9', lw=2)
axes[0].axhline(rainfall['rainfall_mm'].quantile(0.75), color='#e74c3c', ls='--',
    label=f"P75 = {rainfall['rainfall_mm'].quantile(0.75):.0f}mm")
axes[0].set_ylabel('Precipitação (mm)'); axes[0].set_title('Precipitação Mensal — São Luís')
axes[0].legend()

repairs_monthly = repairs.groupby(repairs['order_date'].dt.to_period('M')).size()
repairs_monthly.index = repairs_monthly.index.to_timestamp()
axes[1].bar(repairs_monthly.index, repairs_monthly.values, width=25, color='#e74c3c', alpha=0.7)
axes[1].set_ylabel('Nº de Reparos'); axes[1].set_title('Reparos por Mês')
axes[1].set_xlabel('Data')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "06_temporal_chuva_reparos.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 7. SAZONALIDADE
# ═══════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))
monthly_avg = rainfall.groupby('month')['rainfall_mm'].mean()
repairs['repair_month'] = repairs['order_date'].dt.month
repair_monthly_avg = repairs.groupby('repair_month').size()

ax2 = ax.twinx()
ax.bar(monthly_avg.index, monthly_avg.values, color='#3498db', alpha=0.5, label='Chuva média (mm)')
ax2.plot(repair_monthly_avg.index, repair_monthly_avg.values, 'o-', color='#e74c3c',
    lw=2.5, markersize=8, label='Nº reparos')
ax.set_xlabel('Mês'); ax.set_ylabel('Precipitação (mm)', color='#3498db')
ax2.set_ylabel('Nº de Reparos', color='#e74c3c')
ax.set_title('Sazonalidade: Chuva vs Reparos por Mês')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'])
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "07_sazonalidade.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 8. SCATTER PLOTS — RELAÇÕES COM VARIÁVEL ALVO
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
scatter_vars = [
    ('pavement_age_years', 'Idade do Pavimento (anos)'),
    ('total_repairs', 'Total de Reparos Históricos'),
    ('total_complaints', 'Total de Reclamações'),
    ('load_index_1_10', 'Índice de Carga (1-10)')
]
for ax, (col, label) in zip(axes.flat, scatter_vars):
    data = analytical.dropna(subset=[col, 'risk_score'])
    ax.scatter(data[col], data['risk_score'], c=data['risk_score'],
        cmap='RdYlGn_r', s=100, edgecolors='white', linewidth=0.5, zorder=5)
    if len(data) > 2:
        z = np.polyfit(data[col], data['risk_score'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(data[col].min(), data[col].max(), 100)
        ax.plot(x_line, p(x_line), '--', color='#e74c3c', lw=2, alpha=0.7)
        r, pval = stats.pearsonr(data[col], data['risk_score'])
        ax.text(0.05, 0.95, f'r={r:.3f}\np={pval:.3f}', transform=ax.transAxes,
            va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_xlabel(label); ax.set_ylabel('Risk Score')
    ax.set_title(f'{label} vs Risk Score')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "08_scatter_preditores.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 9. ANÁLISE DE CUSTOS
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cost_seg = analytical[analytical['total_repair_cost']>0].sort_values('total_repair_cost', ascending=True)
axes[0].barh(cost_seg['street_name']+' ('+cost_seg['bairro']+')',
    cost_seg['total_repair_cost'], color=sns.color_palette("YlOrRd", len(cost_seg)))
axes[0].set_xlabel('Custo Total (R$)'); axes[0].set_title('Custo Acumulado de Reparos por Segmento')
for i, v in enumerate(cost_seg['total_repair_cost']):
    axes[0].text(v+100, i, f'R${v:,.0f}', va='center', fontsize=9)

repair_types = repairs['repair_type'].value_counts()
axes[1].pie(repair_types, labels=repair_types.index, autopct='%1.0f%%',
    colors=COLORS, startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Distribuição por Tipo de Reparo')

plt.tight_layout()
plt.savefig(os.path.join(OUT, "09_custos.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 10. FLOOD ZONE vs RISCO
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

flood_groups = analytical.groupby('flood_zone')['risk_score'].apply(list)
flood_data = [flood_groups.get(0, []), flood_groups.get(1, [])]
bp = axes[0].boxplot(flood_data, labels=['Fora de Baixada','Zona de Alagamento'],
    patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#2ecc71'); bp['boxes'][1].set_facecolor('#e74c3c')
for b in bp['boxes']: b.set_alpha(0.7)
axes[0].set_ylabel('Risk Score'); axes[0].set_title('Risco: Zona de Alagamento vs Normal')

# Teste estatístico
if len(flood_data[0])>1 and len(flood_data[1])>1:
    t_stat, p_val = stats.mannwhitneyu(flood_data[0], flood_data[1], alternative='two-sided')
    axes[0].text(0.5, 0.95, f'Mann-Whitney p={p_val:.4f}', transform=axes[0].transAxes,
        ha='center', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Reclamações por canal
channels = complaints['channel'].value_counts()
axes[1].bar(channels.index, channels.values, color=COLORS[:len(channels)], edgecolor='white')
axes[1].set_title('Reclamações por Canal'); axes[1].set_ylabel('Quantidade')
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(OUT, "10_flood_reclamacoes.png"), bbox_inches='tight')
plt.close()

# ═══════════════════════════════════════════════════════════
# 11. MAPA DE SÃO LUÍS (Folium)
# ═══════════════════════════════════════════════════════════
try:
    from folium.plugins import MarkerCluster

    m = folium.Map(location=[-2.53, -44.30], zoom_start=13,
        tiles='CartoDB positron', control_scale=True)

    # Cores por categoria de risco
    risk_colors = {'critico': 'red', 'alto': 'orange', 'medio': 'blue', 'baixo': 'green'}

    # Adicionar segmentos como linhas no mapa
    for _, row in segments.iterrows():
        seg_id = row['segment_id']
        risk_row = risk[risk['segment_id'] == seg_id]
        if len(risk_row) > 0:
            cat = risk_row.iloc[0]['risk_category']
            score = risk_row.iloc[0]['risk_score_0_100']
            action = risk_row.iloc[0]['recommended_action']
        else:
            cat, score, action = 'baixo', 0, 'N/A'

        color = risk_colors.get(cat, 'gray')
        weight = 8 if cat in ('critico','alto') else 4

        popup_html = f"""
        <div style="font-family:Arial;width:250px">
        <h4 style="margin:0;color:{color}">{row['street_name']}</h4>
        <b>Bairro:</b> {row['bairro']}<br>
        <b>Zona:</b> {row['zone']}<br>
        <b>Pavimento:</b> {row['pavement_type']}<br>
        <b>Score:</b> {score} ({cat.upper()})<br>
        <b>Ação:</b> {action.replace('_',' ')}<br>
        <b>Alagamento:</b> {row['flood_zone']}
        </div>"""

        folium.PolyLine(
            [[row['lat_start'], row['lon_start']], [row['lat_end'], row['lon_end']]],
            color=color, weight=weight, opacity=0.8,
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"{row['street_name']} — {cat.upper()} ({score})"
        ).add_to(m)

    # Adicionar marcadores de reclamações
    complaint_cluster = MarkerCluster(name="Reclamações Cidadãs").add_to(m)
    for _, row in complaints.iterrows():
        if pd.notna(row.get('lat')) and pd.notna(row.get('lon')):
            sev = row.get('severity_reported', 'N/A')
            icon_color = 'red' if sev == 'muito_grave' else 'orange' if sev == 'grave' else 'blue'
            folium.Marker(
                [row['lat'], row['lon']],
                popup=f"{row['complaint_type']} — {sev}<br>{row['report_date']}",
                icon=folium.Icon(color=icon_color, icon='exclamation-sign', prefix='glyphicon')
            ).add_to(complaint_cluster)

    # Legenda
    legend_html = """
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;
        background:white;padding:15px;border-radius:8px;box-shadow:0 2px 6px rgba(0,0,0,0.3);
        font-family:Arial;font-size:13px">
    <b>Risco Viário — São Luís</b><br>
    <i style="background:red;width:18px;height:4px;display:inline-block;margin-right:5px"></i> Crítico<br>
    <i style="background:orange;width:18px;height:4px;display:inline-block;margin-right:5px"></i> Alto<br>
    <i style="background:blue;width:18px;height:4px;display:inline-block;margin-right:5px"></i> Médio<br>
    <i style="background:green;width:18px;height:4px;display:inline-block;margin-right:5px"></i> Baixo<br>
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    folium.LayerControl().add_to(m)
    map_path = os.path.join(OUT, "11_mapa_sao_luis.html")
    m.save(map_path)

except ImportError:

# ═══════════════════════════════════════════════════════════
# 12. TESTES DE HIPÓTESES
# ═══════════════════════════════════════════════════════════

# H1: Segmentos em zona de alagamento têm mais reparos
flood_yes = analytical[analytical['flood_zone']==1]['total_repairs']
flood_no = analytical[analytical['flood_zone']==0]['total_repairs']
if len(flood_yes)>0 and len(flood_no)>0:
    u_stat, p = stats.mannwhitneyu(flood_yes, flood_no, alternative='greater')

# H2: Pavimento mais velho → maior risco
valid = analytical.dropna(subset=['pavement_age_years','risk_score'])
if len(valid)>2:
    r, p = stats.pearsonr(valid['pavement_age_years'], valid['risk_score'])

# H3: Chuva concentrada no 1º semestre (sazonalidade)
sem1 = rainfall[rainfall['month'].between(1,6)]['rainfall_mm']
sem2 = rainfall[rainfall['month'].between(7,12)]['rainfall_mm']
t_stat, p = stats.ttest_ind(sem1, sem2)

# H4: Reclamações graves concentradas em segmentos de alto risco
high = analytical[analytical['risk_category'].isin(['alto','critico'])]['total_complaints']
low = analytical[analytical['risk_category'].isin(['baixo','medio'])]['total_complaints']
if len(high)>0 and len(low)>0:
    u_stat, p = stats.mannwhitneyu(high, low, alternative='greater')

# ═══════════════════════════════════════════════════════════
# 13. RESUMO DE ANOMALIAS
# ═══════════════════════════════════════════════════════════

# Segmentos com custo desproporcional
if analytical['total_repair_cost'].sum() > 0:
    top_cost = analytical.nlargest(3, 'total_repair_cost')[['street_name','bairro','total_repair_cost','total_repairs']]
    for _, r in top_cost.iterrows():

# Segmento com recorrência alta
recurrent = analytical[analytical['recurrence_rate']>=1.0]
if len(recurrent)>0:
    for _, r in recurrent.iterrows():

## 3. Modelagem Preditiva (Machine Learning)

In [ ]:
import pandas as pd
import numpy as np
"""
Projeto: Preditor de Falhas em Vias - São Luís / MA
Etapa:   Modelagem e Machine Learning (Modelling)
"""

matplotlib.use('Agg')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# 1. CARREGAR DADOS
BASE = os.path.dirname(os.path.abspath(__file__))
PARENT = os.path.dirname(BASE)
OUT = os.path.join(BASE, "ml_outputs")
os.makedirs(OUT, exist_ok=True)


data_path = os.path.join(PARENT, "06_ml_training_dataset.csv")
df = pd.read_csv(data_path)

# 2. PRÉ-PROCESSAMENTO

# Tratar valores 'NA' na coluna years_since_last_repair
df['years_since_last_repair'] = df['years_since_last_repair'].replace('NA', np.nan)
df['years_since_last_repair'] = pd.to_numeric(df['years_since_last_repair'])

# Preencher nan em years_since_last_repair com a idade do pavimento
df['years_since_last_repair'].fillna(df['road_age_years'], inplace=True)

# Remover colunas que não são features numéricas ou são labels vazamentos
drop_cols = ['segment_id', 'year', 'month', 'notes', 'LABEL_failed_next_30d', 'LABEL_failed_next_90d', 'repair_cost_total_brl', 'cumulative_repairs']
X = df.drop(columns=drop_cols, errors='ignore')

# Target principal: falha nos próximos 90 dias (mais dados para modelo de manutenção preventiva)
y = df['LABEL_failed_next_90d']


# Divisão Treino e Teste (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Escalonamento e imputação (para os poucos nulos que podem sobrar)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# 3. SELEÇÃO E TREINAMENTO DOS MODELOS BÁSICOS (BASELINE)

models = {
    "Regressão Logística": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    results.append({
        "Model": name,
        "ROC-AUC CV Mean": cv_scores.mean(),
        "ROC-AUC CV Std": cv_scores.std()
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC CV Mean', ascending=False)

# O melhor modelo é escolhido baseado no AUC
best_model_name = results_df.iloc[0]['Model']

# 4. TUNING DE HIPERPARÂMETROS DO XGBOOST (ASSUMINDO QUE SEJA O MELHOR)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Em datasets muito pequenos (como nosso sample), o GridSearch pode não variar muito. 
# cv=3 para evitar fold muito pequeno
grid_search = GridSearchCV(xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
grid_search.fit(X_train_scaled, y_train)

best_xgb = grid_search.best_estimator_
for k, v in grid_search.best_params_.items():

# 5. AVALIAÇÃO FINAL NO CONJUNTO DE TESTE

# Predições
y_pred = best_xgb.predict(X_test_scaled)
y_prob = best_xgb.predict_proba(X_test_scaled)[:, 1]

# Métricas
auc_test = roc_auc_score(y_test, y_prob)

# Gerar Matriz de Confusão e Curva ROC em imagem
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusão (Teste)')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='red', label=f'ROC curve (AUC = {auc_test:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
axes[1].set_xlabel('Taxa Falsos Positivos')
axes[1].set_ylabel('Taxa Verdadeiros Positivos')
axes[1].set_title('Curva ROC')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(OUT, "01_avaliacao_modelo.png"))
plt.close()

# 6. IMPORTÂNCIA DAS VARIÁVEIS (FEATURE IMPORTANCE)

importances = best_xgb.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10, 6))
plt.title("Feature Importances (XGBoost)")
sns.barplot(x=importances[indices], y=[features[i] for i in indices], palette='viridis')
plt.xlabel("Importância Relativa")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "02_feature_importance.png"))
plt.close()

for i in indices[:5]:

## 4. Avaliação de Negócio (ROI)

In [ ]:
import pandas as pd
import numpy as np
"""
Projeto: Preditor de Falhas em Vias - São Luís / MA
Etapa:   Avaliação e Interpretação (Business Evaluation)
"""

matplotlib.use('Agg')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# 1. CARREGAR DADOS E RETREINAR O MODELO PARA OBTER PREDIÇÕES
BASE = os.path.dirname(os.path.abspath(__file__))
PARENT = os.path.dirname(BASE)
OUT = os.path.join(BASE, "ml_outputs")
os.makedirs(OUT, exist_ok=True)

data_path = os.path.join(PARENT, "06_ml_training_dataset.csv")
df = pd.read_csv(data_path)

df['years_since_last_repair'] = df['years_since_last_repair'].replace('NA', np.nan)
df['years_since_last_repair'] = pd.to_numeric(df['years_since_last_repair'])
df['years_since_last_repair'].fillna(df['road_age_years'], inplace=True)

drop_cols = ['segment_id', 'year', 'month', 'notes', 'LABEL_failed_next_30d', 'LABEL_failed_next_90d', 'repair_cost_total_brl', 'cumulative_repairs']
X = df.drop(columns=drop_cols, errors='ignore')
y = df['LABEL_failed_next_90d']

# Salvamos os índices para podermos referenciar depois os custos
X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
    X, y, df.index, test_size=0.3, stratify=y, random_state=42)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Melhor modelo
best_xgb = XGBClassifier(
    learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8,
    use_label_encoder=False, eval_metric='logloss', random_state=42
)
best_xgb.fit(X_train_scaled, y_train)

y_pred = best_xgb.predict(X_test_scaled)
y_prob = best_xgb.predict_proba(X_test_scaled)[:, 1]

# 2. MÉTRICAS TÉCNICAS (PERFORMANCE)
auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)


# 3. RESULTADOS DE NEGÓCIO (BUSINESS INTERPRETATION)

# Custo médio derivado do relatório executivo e dicionário de dados
# Reparo preventivo médio: ~ R$ 3.500
# Reparo emergencial (falha não prevista): ~ R$ 12.000
# Custo de vistoriar um alarme falso (inspeção): ~ R$ 200

COST_PREVENTIVE = 3500
COST_EMERGENCY = 12000
COST_INSPECTION_FALSE_ALARM = 200

# Cálculo de Custos do Cenário ATUAL (100% Reativo nas vias que falharam)
# Vias que realmente falharam na amostra de teste
total_falhas_reais = np.sum(y_test == 1)
custo_cenario_atual = total_falhas_reais * COST_EMERGENCY

# Cálculo de Custos com o MODELO PREDITIVO
# Verdadeiros Positivos (Detectou preventivamente -> Custo Preventivo)
custo_vp = cm[1][1] * COST_PREVENTIVE
# Falsos Negativos (O modelo errou e não detectou -> Custo Emergencial)
custo_fn = cm[1][0] * COST_EMERGENCY
# Falsos Positivos (Modelo previu falha, vistoria foi feita, mas estava bom -> Custo de Inspeção)
custo_fp = cm[0][1] * COST_INSPECTION_FALSE_ALARM

custo_cenario_ia = custo_vp + custo_fn + custo_fp
economia_reais = custo_cenario_atual - custo_cenario_ia
economia_pct = (economia_reais / custo_cenario_atual) * 100 if custo_cenario_atual > 0 else 0


# Projetando para toda a cidade de São Luís (~ 2000 trechos críticos)
# Vamos escalar o resultado da amostra (que tinha 12 segmentos)
escala_cidade = 2000 / len(y_test)
economia_cidade_projetada = economia_reais * escala_cidade


# Gerando gráfico de ROI (Return on Investment)
fig, ax = plt.subplots(figsize=(8, 6))
labels = ['Custo Atual (Reativo)', 'Custo c/ IA (Preditivo)']
valores = [custo_cenario_atual, custo_cenario_ia]
cores = ['#e74c3c', '#2ecc71']

bars = ax.bar(labels, valores, color=cores, width=0.5)
ax.set_ylabel('Custo em R$ (Milhares)')
ax.set_title('Simulação de Custo em Manutenção (Amostra de Teste)')

for b in bars:
    altura = b.get_height()
    ax.text(b.get_x() + b.get_width()/2., altura + 1000,
            f'R$ {altura:,.0f}', ha='center', fontweight='bold')

# Adiciona linha de economia
if economia_reais > 0:
    ax.annotate(f'Economia:\nR$ {economia_reais:,.0f}', 
                xy=(1, custo_cenario_ia), 
                xytext=(0.5, custo_cenario_atual),
                arrowprops=dict(facecolor='black', shrink=0.05),
                ha='center', fontsize=12, fontweight='bold', color='green',
                bbox=dict(boxstyle="round,pad=0.3", fc="lightgreen", ec="g", lw=2))

plt.tight_layout()
plt.savefig(os.path.join(OUT, "03_business_impact.png"))
plt.close()